# Original Reading List Analysis (Read-Only)

This notebook reads the original Reading List database and extracts all Summary Table rows from paper subpages for analysis only.

Output: `analysis_dfs` (dict of 5 DataFrames)
1. Reading List + Page Fields
2. Concatenated Summary Table Pred
3. Concatenated Summary Table Desc
4. Concatenated Summary Table KT
5. Concatenated Summary Table Rec

In [2]:
# Load Notion_Zotero environment and check credentials for analysis
import os
from dotenv import load_dotenv
import pandas as pd

# Notion_Zotero expects NOTION_API_KEY and NOTION_DATABASE_ID in .env
load_dotenv()

NOTION_API_KEY = os.getenv("NOTION_API_KEY", "").strip().strip('\"').strip("'")
NOTION_DATABASE_ID = os.getenv("NOTION_DATABASE_ID", "").strip().strip('\"').strip("'")

missing = []
if not NOTION_API_KEY:
    missing.append("NOTION_API_KEY")
if not NOTION_DATABASE_ID:
    missing.append("NOTION_DATABASE_ID")

if missing:
    raise ValueError(f"Missing env vars: {', '.join(missing)}")

print("Credentials OK")
print("Notion Reading List DB ID:", NOTION_DATABASE_ID)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# 1) Use Notion_Zotero CLI to export and analyze the reading list
import subprocess
import sys
from pathlib import Path

# Step 1: Export Notion database to fixtures/reading_list (if not already done)
export_cmd = [
    sys.executable, "-m", "notion_zotero.scripts.export_reading_list",
    "--db-id", NOTION_DATABASE_ID,
    "--out", "fixtures/reading_list"
 ]
print("Exporting Notion reading list to fixtures/reading_list ...")
subprocess.run(export_cmd, check=True)

# Step 2: Parse fixtures into canonical bundles
parse_cmd = [
    "notion-zotero", "parse-fixtures",
    "--input", "fixtures/reading_list",
    "--out", "fixtures/canonical"
 ]
print("Parsing fixtures/reading_list into canonical bundles ...")
subprocess.run(parse_cmd, check=True)

# Step 3: Merge canonical bundles
merge_cmd = [
    "notion-zotero", "merge-canonical",
    "--input", "fixtures/canonical",
    "--out", "fixtures/canonical_merged.json"
 ]
print("Merging canonical bundles ...")
subprocess.run(merge_cmd, check=True)

# Step 4: Load merged canonical data for analysis
import json
with open("fixtures/canonical_merged.json", "r", encoding="utf-8") as f:
    canonical_data = json.load(f)
df_canonical = pd.DataFrame(canonical_data)
print(f"Loaded {len(df_canonical)} canonical references.")
df_canonical.head()

In [ ]:
# 2) Example analysis: summary statistics and inspection
print("Column names:", df_canonical.columns.tolist())
print("Sample rows:")
display(df_canonical.head(5))

# Example: count by publication year if available
if 'year' in df_canonical.columns:
    print("\nReferences by year:")
    display(df_canonical['year'].value_counts().sort_index())

# Example: count by reference type if available
if 'type' in df_canonical.columns:
    print("\nReferences by type:")
    display(df_canonical['type'].value_counts())

In [ ]:
# 3) Concatenate summary rows per task (keep duplicates by design)
summary_collectors = {task: [] for task in TASK_ORDER}
errors = []

for record in original_records:
    try:
        extracted = extract_summary_rows_for_record(record)
        for task in TASK_ORDER:
            summary_collectors[task].extend(extracted.get(task, []))
    except Exception as exc:
        errors.append({
            "source_paper_title": _paper_title(record),
            "source_page_id": record.get("notion_page_id"),
            "error": str(exc),
        })

df_pred = pd.DataFrame(summary_collectors["PRED"]).fillna("")
df_desc = pd.DataFrame(summary_collectors["DESC"]).fillna("")
df_kt = pd.DataFrame(summary_collectors["KT"]).fillna("")
df_rec = pd.DataFrame(summary_collectors["REC"]).fillna("")

analysis_dfs = {
    "Reading List + Page Fields": df_reading_list,
    "Concatenated Summary Table Pred": df_pred,
    "Concatenated Summary Table Desc": df_desc,
    "Concatenated Summary Table KT": df_kt,
    "Concatenated Summary Table Rec": df_rec,
}

print("analysis_dfs ready with 5 DataFrames:")
for key, df in analysis_dfs.items():
    print(f"  - {key}: {len(df)} rows")

if errors:
    print(f"\nRows with extraction errors: {len(errors)}")
    display(pd.DataFrame(errors).head(10))

In [ ]:
# 4) Quick inspection samples
display(analysis_dfs["Reading List + Page Fields"].head(3))
display(analysis_dfs["Concatenated Summary Table Pred"].head(3))
display(analysis_dfs["Concatenated Summary Table Desc"].head(3))
display(analysis_dfs["Concatenated Summary Table KT"].head(3))
display(analysis_dfs["Concatenated Summary Table Rec"].head(3))

## 5) Standard Cleaning Layer (Raw -> Clean)

This section applies a standard normalization pass to all extracted tables.

Outputs created:
- `raw_tables` (immutable snapshot from `analysis_dfs`)
- `analysis_dfs_clean` (normalized tables)
- `normalization_log` (light summary of transformations)

In [ ]:
import re

# -------------------------------
# 0) Raw snapshot (do not mutate)
# -------------------------------
raw_tables = {k: v.copy() for k, v in analysis_dfs.items()}

# -------------------------------
# 1) Rules
# -------------------------------
GLOBAL_COLUMN_ALIASES = {
    "thereotical model": "Theoretical Model",
    "theoretical model": "Theoretical Model",
    "data sources": "Data Sources",
    "data source": "Data Sources",
    "recommendation types": "Recommendation Types",
    "performance metric: best model": "Performance Metric: Best Model",
}

TABLE_COLUMN_ALIASES = {
    "Reading List + Page Fields": {
        "keywords/type": "Keywords/Type",
    },
    "Concatenated Summary Table Rec": {
        "recommender system type": "Recommender System Type",
    },
}

GENERIC_VALUE_MAP = {
    "none": "Not Applicable",
    "none applicable": "Not Applicable",
    "not applicable": "Not Applicable",
    "n/a": "Not Applicable",
    "na": "Not Applicable",
    "elearning": "E-Learning",
    "e-learning": "E-Learning",
}

RECOMMENDER_TYPE_MAP = {
    "collaborative filtering": "Collaborative Filtering",
    "collaborative-filtering": "Collaborative Filtering",
    "content based filtering": "Content-Based Filtering",
    "content-based filtering": "Content-Based Filtering",
    "content-based fiiltering": "Content-Based Filtering",
    "knowledge based": "Knowledge-Based Filtering",
    "knowledge-based": "Knowledge-Based Filtering",
    "hybrid": "Hybrid Recommender",
    "hybrid recommender": "Hybrid Recommender",
    "reinforcement learning": "Reinforcement Learning Recommender",
    "rl recommender": "Reinforcement Learning Recommender",
    "sequence aware": "Sequence-Aware Recommender",
    "sequence-aware": "Sequence-Aware Recommender",
    "graph based": "Graph-Based Recommender",
    "graph-based": "Graph-Based Recommender",
    "llm": "LLM-Augmented Recommender",
    "llm augmented": "LLM-Augmented Recommender",
    "llm-augmented": "LLM-Augmented Recommender",
}

TYPO_FIXES = {
    r"\bFiiltering\b": "Filtering",
    r"\bExerrcise\b": "Exercise",
    r"\baprroach\b": "approach",
    r"\bThereotical\b": "Theoretical",
    r"\bMAchine\b": "Machine",
}

FREE_TEXT_COLUMNS = {
    "Search Strategy",
    "Comments",
    "Limitations",
    "Context",
    "Preprocessing Details",
    "Cluster Description",
    "Implications",
    "Evaluation",
    "Performance Metric: Best Model",
}

METRIC_NAME_MAP = {
    "acc": "Accuracy",
    "accuracy": "Accuracy",
    "prec": "Precision",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1",
    "f1 score": "F1",
    "f1 macro": "F1-Macro",
    "f1 micro": "F1-Micro",
    "f1 weighted": "F1-Weighted",
    "auc": "AUC",
    "auroc": "AUC",
    "auc roc": "AUC",
    "mcc": "MCC",
    "mae": "MAE",
    "mse": "MSE",
    "rmse": "RMSE",
    "r2": "R2",
    "hit-rate@10": "HitRate@10",
    "ndcg@10": "NDCG@10",
}

PROPORTION_METRICS = {
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "F1-Macro",
    "F1-Micro",
    "F1-Weighted",
    "AUC",
    "MCC",
    "HitRate@10",
    "NDCG@10",
}

ACRONYM_WORDS = {"LAK", "LMS", "MOOC", "AI", "LLM", "KT", "RS"}

SEARCH_STRATEGY_TERM_MAP = {
    "machine learning": "Machine Learning",
    "learning analytics": "Learning Analytics",
    "student performance": "Student Performance",
    "early warning system": "Early Warning System",
    "learning management system": "Learning Management System",
    "knowledge tracing": "Knowledge Tracing",
    "recommender systems": "Recommender Systems",
    "education": "Education",
    "higher education": "Higher Education",
    "group formation": "Group Formation",
    "collaborative learning": "Collaborative Learning",
    "clickstream data": "Clickstream Data",
    "transformer": "Transformer",
    "semi-supervised learning": "Semi-Supervised Learning",
    "at-risk students": "At-Risk Students",
    "learning strategies": "Learning Strategies",
    "lak proceedings": "LAK Proceedings",
    "multi-modal": "Multi-Modal",
}

# -------------------------------
# 2) Helpers
# -------------------------------

def _norm_key(text: str) -> str:
    text = str(text).replace("_", " ").replace("-", " ")
    return re.sub(r"\s+", " ", text).strip().lower()


def _clean_whitespace(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s*\|\s*", " | ", text)
    text = re.sub(r"\s*;\s*", "; ", text)
    return text.strip()


def _apply_typos(text: str) -> str:
    out = text
    for pattern, repl in TYPO_FIXES.items():
        out = re.sub(pattern, repl, out, flags=re.IGNORECASE)
    return out


def _normalize_text_cell(v):
    if not isinstance(v, str):
        return v
    out = _clean_whitespace(v)
    out = _apply_typos(out)
    mapped = GENERIC_VALUE_MAP.get(_norm_key(out), out)
    return mapped


def _normalize_term_case(term: str) -> str:
    words = [w for w in re.split(r"\s+", term.strip()) if w]
    out = []
    for w in words:
        if w.upper() in ACRONYM_WORDS:
            out.append(w.upper())
        elif w.lower() == "and":
            out.append("AND")
        else:
            out.append(w[:1].upper() + w[1:].lower())
    return " ".join(out)


def _normalize_search_strategy(value):
    if not isinstance(value, str):
        return value

    s = _normalize_text_cell(value)
    # Replace curly quotes with straight equivalents
    s = s.replace("\u201c", '"').replace("\u201d", '"').replace("\u2019", "'")
    s = s.replace("\n", " ").strip()

    parts = [p.strip() for p in re.split(r"\s+AND\s+", s, flags=re.IGNORECASE) if p.strip()]
    if not parts:
        return s

    normalized_terms = []
    for p in parts:
        p = p.strip(" \"'\u201c\u201d\u2018\u2019`()")
        p = _normalize_text_cell(p)

        key = _norm_key(p)
        canon = SEARCH_STRATEGY_TERM_MAP.get(key)
        if canon is None:
            canon = _normalize_term_case(p)

        normalized_terms.append(f'"{canon}"')

    deduped = list(dict.fromkeys(normalized_terms))
    return " AND ".join(deduped)


def _canonical_metric_name(name: str) -> str:
    n = _norm_key(name)
    return METRIC_NAME_MAP.get(n, " ".join(w.capitalize() for w in n.split()))


def _metric_to_decimal(metric_name: str, value: float, had_percent: bool) -> float:
    if had_percent:
        value = value / 100.0
    elif metric_name in PROPORTION_METRICS and value > 1.0 and value <= 100.0:
        value = value / 100.0

    if metric_name in PROPORTION_METRICS:
        value = max(0.0, min(1.0, value))
    return value


def _split_metric_segments(text: str) -> list:
    """Split a metric string by top-level commas or semicolons (ignoring those inside brackets)."""
    segments = []
    depth = 0
    current = []
    for ch in text:
        if ch in "([{":
            depth += 1
            current.append(ch)
        elif ch in ")]}":
            depth -= 1
            current.append(ch)
        elif ch in ",;" and depth == 0:
            seg = "".join(current).strip()
            if seg:
                segments.append(seg)
            current = []
        else:
            current.append(ch)
    if current:
        seg = "".join(current).strip()
        if seg:
            segments.append(seg)
    return segments


# Matches one metric segment: "MetricName: value[%] [- ModelName]"
# Value may be a plain number or a bracketed range like [0.65-0.79]
_SEG_RE = re.compile(
    r"^([A-Za-z][A-Za-z0-9@#%+_/\- ]{0,40}?)"              # metric name (non-greedy)
    r"\s*[:=]\s*"
    r"(\[?\s*-?\d+(?:\.\d+)?(?:\s*[-\u2013]\s*\d+(?:\.\d+)?)?\s*\]?)"  # value or [lo-hi]
    r"\s*(%?)"                                               # optional percent
    r"(?:\s*[-\u2013]\s*([A-Za-z][A-Za-z0-9 /+\-\.]*?))?"  # optional " - ModelName"
    r"\s*$",
    re.IGNORECASE,
)


def _normalize_metric_text(text: str) -> str:
    """
    Normalize a performance metric cell to a consistent key=value format.

    Examples:
      "Accuracy: 0.826 - Stacking, F1-Score: 0.792"  ->  "Accuracy=0.826 [Stacking]; F1=0.792"
      "{AUC: [0,65-0,79] - SVM}"                      ->  "AUC=0.650-0.790 [SVM]"
      "Accuracy: 82.6%"                               ->  "Accuracy=0.826"
    """
    if not isinstance(text, str) or not text.strip():
        return text

    text = _normalize_text_cell(text)

    # Normalize European decimal comma inside numbers (0,65 -> 0.65)
    text = re.sub(r"(?<=\d),(?=\d)", ".", text)

    # Strip outer braces wrapping the whole value
    stripped = text.strip()
    if stripped.startswith("{") and stripped.endswith("}"):
        stripped = stripped[1:-1].strip()

    segments = _split_metric_segments(stripped)

    pairs = []
    unparsed = []

    for seg in segments:
        seg = seg.strip(" {}[]")
        m = _SEG_RE.match(seg)
        if not m:
            if seg:
                unparsed.append(seg)
            continue

        raw_name = m.group(1).strip(" -_")
        raw_value_str = m.group(2).strip(" []")
        had_percent = bool(m.group(3))
        model_name = m.group(4).strip() if m.group(4) else None

        metric_name = _canonical_metric_name(raw_name)

        num_parts = re.findall(r"-?\d+(?:\.\d+)?", raw_value_str)
        if len(num_parts) >= 2:
            lo = _metric_to_decimal(metric_name, float(num_parts[0]), had_percent)
            hi = _metric_to_decimal(metric_name, float(num_parts[1]), had_percent)
            value_str = f"{lo:.3f}-{hi:.3f}"
        elif len(num_parts) == 1:
            val = _metric_to_decimal(metric_name, float(num_parts[0]), had_percent)
            value_str = f"{val:.4f}".rstrip("0").rstrip(".")
        else:
            value_str = raw_value_str

        entry = f"{metric_name}={value_str}"
        if model_name:
            entry += f" [{model_name}]"
        pairs.append(entry)

    if not pairs:
        return text  # nothing matched, return original as-is

    out = "; ".join(pairs)
    if unparsed:
        out += "; Notes=" + ", ".join(unparsed)
    return out


def _canonicalize_columns(df: pd.DataFrame, table_name: str):
    alias = {}
    alias.update(GLOBAL_COLUMN_ALIASES)
    alias.update(TABLE_COLUMN_ALIASES.get(table_name, {}))

    rename_map = {}
    seen_targets = set(df.columns)
    for col in df.columns:
        key = _norm_key(col)
        target = alias.get(key, col)
        if target != col:
            if target in seen_targets and target not in rename_map.values():
                continue
            rename_map[col] = target
    return df.rename(columns=rename_map), rename_map


def _coalesce_alias_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "Data Sources" in out.columns and "Data sources" in out.columns:
        out["Data Sources"] = out["Data Sources"].where(
            out["Data Sources"].astype(str).str.strip() != "",
            out["Data sources"],
        )
        out = out.drop(columns=["Data sources"])
    return out


def standard_clean_table(df: pd.DataFrame, table_name: str):
    out, rename_map = _canonicalize_columns(df, table_name)
    out = _coalesce_alias_columns(out)

    text_updates = 0
    search_strategy_updates = 0

    for col in out.columns:
        if out[col].dtype != object:
            continue

        original = out[col].copy()
        out[col] = out[col].map(_normalize_text_cell)

        if table_name == "Reading List + Page Fields" and col == "Search Strategy":
            out[col] = out[col].map(_normalize_search_strategy)
            search_strategy_updates += int((original != out[col]).sum())

        text_updates += int((original != out[col]).sum())

    metric_updates = 0
    metric_cols = [
        c
        for c in out.columns
        if _norm_key(c) in {_norm_key("Performance Metric: Best Model"), _norm_key("Evaluation")}
        or "metric" in _norm_key(c)
    ]
    for mcol in metric_cols:
        original = out[mcol].copy()
        out[mcol] = out[mcol].map(_normalize_metric_text)
        metric_updates += int((original != out[mcol]).sum())

    if table_name == "Concatenated Summary Table Rec" and "Recommender System Type" in out.columns:
        original = out["Recommender System Type"].copy()

        def _norm_rec(v):
            if not isinstance(v, str):
                return v
            s = _normalize_text_cell(v)
            n = _norm_key(s)
            mapped = RECOMMENDER_TYPE_MAP.get(n)
            if mapped:
                return mapped
            if "hybrid" in n:
                return "Hybrid Recommender"
            if "content" in n and "filter" in n:
                return "Content-Based Filtering"
            if "collaborative" in n:
                return "Collaborative Filtering"
            if "knowledge" in n:
                return "Knowledge-Based Filtering"
            if "reinforcement" in n or n.startswith("rl"):
                return "Reinforcement Learning Recommender"
            if "sequence" in n:
                return "Sequence-Aware Recommender"
            if "graph" in n:
                return "Graph-Based Recommender"
            if "llm" in n:
                return "LLM-Augmented Recommender"
            return s

        out["Recommender System Type"] = out["Recommender System Type"].map(_norm_rec)
        recommender_updates = int((original != out["Recommender System Type"]).sum())
    else:
        recommender_updates = 0

    log = {
        "table": table_name,
        "rows_before": len(df),
        "rows_after": len(out),
        "renamed_columns": rename_map,
        "text_updates": text_updates,
        "search_strategy_updates": search_strategy_updates,
        "metric_updates": metric_updates,
        "recommender_updates": recommender_updates,
    }
    return out, log

# -------------------------------
# 3) Apply standard cleaning to all tables
# -------------------------------
analysis_dfs_clean = {}
logs = []

for table_name, raw_df in raw_tables.items():
    cleaned_df, table_log = standard_clean_table(raw_df, table_name)
    analysis_dfs_clean[table_name] = cleaned_df
    logs.append(table_log)

normalization_log = pd.DataFrame(logs)

print("Standard cleaning complete.")
print("Raw tables available in raw_tables")
print("Clean tables available in analysis_dfs_clean")
display(normalization_log[["table", "text_updates", "metric_updates", "recommender_updates"]])

In [ ]:
# Quick sanity check for _normalize_metric_text fixes
_tests = [
    # (input, expected_output_contains)
    ("{Accuracy: 0.826 - Stacking, Precision: 0.805 - SVM, Recall: 0.825, F1-Score: 0.792}",
     ["Accuracy=0.826 [Stacking]", "Precision=0.805 [SVM]", "Recall=0.825", "F1=0.792"]),
    ("{AUC: [0,65-0,79] - SVM}",
     ["AUC=0.650-0.790 [SVM]"]),
    ("Accuracy: 82.6%",
     ["Accuracy=0.826"]),
    ("AUC: 0.85",
     ["AUC=0.85"]),
]

all_ok = True
for inp, expected_parts in _tests:
    result = _normalize_metric_text(inp)
    ok = all(p in result for p in expected_parts)
    status = "OK" if ok else "FAIL"
    if not ok:
        all_ok = False
    print(f"[{status}]  {inp!r}")
    print(f"       -> {result!r}")

print("\nAll tests passed." if all_ok else "\nSome tests FAILED - check output above.")

In [ ]:
analysis_dfs_clean['Reading List + Page Fields']['Search Strategy'].unique()

In [ ]:
# Export Reading List to fixtures/reading_list (HTTP fallback)
import os, sys, time, json
from pathlib import Path
from dotenv import load_dotenv
import httpx

NOTION_ZOTERO = Path(r"C:\Users\ricar\OneDrive - NOVAIMS\PhD\Publications\Literature Review Paper\Literature Review Analysis\Notion_Zotero")
ENV_PATH = NOTION_ZOTERO / ".env"
if ENV_PATH.exists():
    load_dotenv(dotenv_path=str(ENV_PATH))
    print("Loaded .env from", ENV_PATH)
else:
    load_dotenv()  # fallback to current/workspace .env
    print("Loaded .env from default location (fallback)")

OUT_DIR = NOTION_ZOTERO / "fixtures" / "reading_list"
PAGE_OR_DB_ID = "9944e9224f2e44879dab9b3054445af0"  # replace if different

token = os.getenv("NOTION_TOKEN") or os.getenv("NOTION_API_KEY") or os.getenv("NOTION_INTEGRATION_TOKEN")
if not token:
    raise RuntimeError("Set NOTION_TOKEN in the notebook kernel environment before running this cell.")

headers = {"Authorization": f"Bearer {token}", "Notion-Version": "2022-06-28", "Content-Type": "application/json"}


def plain_text(rich):
    if not rich:
        return ""
    parts = []
    for r in rich:
        if isinstance(r, dict):
            if "plain_text" in r:
                parts.append(r["plain_text"])
            elif "text" in r and isinstance(r["text"], dict):
                parts.append(r["text"].get("content", ""))
        elif isinstance(r, str):
            parts.append(r)
    return "".join(parts)


def fetch_blocks_http(block_id):
    out = []
    start = None
    client = httpx.Client(timeout=30)
    while True:
        params = {"start_cursor": start} if start else {}
        url = f"https://api.notion.com/v1/blocks/{block_id}/children"
        resp = client.get(url, headers=headers, params=params)
        resp.raise_for_status()
        data = resp.json()
        out.extend(data.get("results", []))
        if data.get("has_more"):
            start = data.get("next_cursor")
            time.sleep(0.1)
        else:
            break
    return out


def serialize_block(b):
    t = b.get("type")
    text = ""
    if t in ("paragraph", "heading_1", "heading_2", "heading_3", "bulleted_list_item", "numbered_list_item", "to_do"):
        rt = b.get(t, {}).get("rich_text") or b.get(t, {}).get("text") or []
        text = plain_text(rt)
    return {"id": b.get("id"), "type": t, "text": text, "raw": b}


def extract_tables_http(blocks):
    tables = []
    for idx, b in enumerate(blocks):
        if b.get("type") == "table":
            rows = fetch_blocks_http(b.get("id"))
            parsed = []
            for r in rows:
                if r.get("type") == "table_row":
                    cells = r.get("table_row", {}).get("cells", [])
                    parsed.append([plain_text(c) for c in cells])
            heading = None
            for j in range(max(0, idx - 5), idx):
                bt = blocks[j]
                if bt.get("type", "").startswith("heading"):
                    heading = plain_text(bt.get(bt["type"], {}).get("rich_text", []))
                    break
            tables.append({"block_id": b.get("id"), "heading": heading, "rows": parsed, "index": idx})
    return tables


def export_page_http(page_id):
    client = httpx.Client(timeout=30)
    url = f"https://api.notion.com/v1/pages/{page_id}"
    resp = client.get(url, headers=headers)
    resp.raise_for_status()
    page = resp.json()
    blocks = fetch_blocks_http(page_id)
    fixture = {
        "page_id": page_id,
        "title": plain_text(page.get("properties", {}).get("Name", {}).get("title", [])) or page.get("id"),
        "properties": page.get("properties", {}),
        "tables": extract_tables_http(blocks),
        "blocks": [serialize_block(b) for b in blocks],
    }
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
    path = Path(OUT_DIR) / f"{page_id}.json"
    path.write_text(json.dumps(fixture, ensure_ascii=False, indent=2), encoding="utf-8")
    print("WROTE:", path)


def query_database_http(database_id):
    client = httpx.Client(timeout=30)
    url = f"https://api.notion.com/v1/databases/{database_id}/query"
    start = None
    while True:
        payload = {"page_size": 100}
        if start:
            payload["start_cursor"] = start
        resp = client.post(url, headers=headers, json=payload)
        resp.raise_for_status()
        data = resp.json()
        for r in data.get("results", []):
            pid = r.get("id")
            export_page_http(pid)
        if data.get("has_more"):
            start = data.get("next_cursor")
            time.sleep(0.1)
        else:
            break

# Try database query first, fallback to single page retrieval
try:
    query_database_http(PAGE_OR_DB_ID)
except Exception as ex_db:
    print("Database query failed:", ex_db)
    try:
        export_page_http(PAGE_OR_DB_ID)
    except Exception as ex_page:
        raise RuntimeError(f"Both database query and page retrieval failed: {ex_db} / {ex_page}")


In [ ]:
# Convert fixtures -> canonical fixtures
import os, sys, runpy
from pathlib import Path

SCRIPT = Path(r"C:\Users\ricar\OneDrive - NOVAIMS\PhD\Publications\Literature Review Paper\Literature Review Analysis\Notion_Zotero\src\services\reading_list_importer.py")
IN_DIR = Path(r"C:\Users\ricar\OneDrive - NOVAIMS\PhD\Publications\Literature Review Paper\Literature Review Analysis\Notion_Zotero\fixtures\reading_list")
OUT_DIR = Path(r"C:\Users\ricar\OneDrive - NOVAIMS\PhD\Publications\Literature Review Paper\Literature Review Analysis\Notion_Zotero\fixtures\canonical")

old_argv = sys.argv[:]
sys.argv = [str(SCRIPT), "--input", str(IN_DIR), "--out", str(OUT_DIR)]
print("Running importer:", SCRIPT)
runpy.run_path(str(SCRIPT), run_name="__main__")
sys.argv = old_argv
print("Importer finished. Output:", OUT_DIR)
